# Q&A SYSTEM WITH TRANSFORMERS

A Question Answering (Q&A) system built using Transformer-based models. The project focuses on fine-tuning pre-trained models to extract or generate accurate answers from a given text context.


# FINE TUNING STEP

## Choix du model et du dataset:

### model
https://huggingface.co/distilbert/distilbert-base-cased

### datasets
trivia_qa : https://huggingface.co/datasets/mandarjoshi/trivia_qa
squad : https://huggingface.co/datasets/rajpurkar/squad_v2
natural_question : https://huggingface.co/datasets/google-research-datasets/natural_questions

## install dependencies

In [7]:
!pip install transformers
!pip install torch
!pip install datasets
!pip install huggingface_hub
!pip install evaluate

## Dataset load

In [ ]:
HF_TOKEN = "YOUR_HF_TOKEN"

In [24]:
from huggingface_hub import notebook_login

notebook_login()

In [25]:
from datasets import load_dataset

ds = load_dataset("rajpurkar/squad_v2", split="train[:100]")

README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

In [36]:
ds = ds.train_test_split(test_size=0.2)

In [38]:
ds["train"][0]

{'id': '56bf725c3aeaaa14008c9643',
 'title': 'Beyoncé',
 'context': 'A self-described "modern-day feminist", Beyoncé creates songs that are often characterized by themes of love, relationships, and monogamy, as well as female sexuality and empowerment. On stage, her dynamic, highly choreographed performances have led to critics hailing her as one of the best entertainers in contemporary popular music. Throughout a career spanning 19 years, she has sold over 118 million records as a solo artist, and a further 60 million with Destiny\'s Child, making her one of the best-selling music artists of all time. She has won 20 Grammy Awards and is the most nominated woman in the award\'s history. The Recording Industry Association of America recognized her as the Top Certified Artist in America during the 2000s decade. In 2009, Billboard named her the Top Radio Songs Artist of the Decade, the Top Female Artist of the 2000s and their Artist of the Millennium in 2011. Time listed her among the 100

In [39]:
ds['train']['question']

Column(['In which decade did the Recording Industry Association of America recognize Beyonce as the The Top Certified Artist?', 'In which decade did Beyonce become famous?', 'To set the record for Grammys, how many did Beyonce win?', 'How did Beyonce describe herself as a feminist?', 'Which artist did Beyonce marry?'])

In [40]:
ds['train']['context']

Column(['A self-described "modern-day feminist", Beyoncé creates songs that are often characterized by themes of love, relationships, and monogamy, as well as female sexuality and empowerment. On stage, her dynamic, highly choreographed performances have led to critics hailing her as one of the best entertainers in contemporary popular music. Throughout a career spanning 19 years, she has sold over 118 million records as a solo artist, and a further 60 million with Destiny\'s Child, making her one of the best-selling music artists of all time. She has won 20 Grammy Awards and is the most nominated woman in the award\'s history. The Recording Industry Association of America recognized her as the Top Certified Artist in America during the 2000s decade. In 2009, Billboard named her the Top Radio Songs Artist of the Decade, the Top Female Artist of the 2000s and their Artist of the Millennium in 2011. Time listed her among the 100 most influential people in the world in 2013 and 2014. Forb

In [43]:
ds["train"]["answers"]

Column([{'text': ['2000s'], 'answer_start': [736]}, {'text': ['late 1990s'], 'answer_start': [276]}, {'text': ['six'], 'answer_start': [565]}, {'text': ['modern-day feminist'], 'answer_start': [18]}, {'text': ['Jay Z'], 'answer_start': [369]}])

## Prepocessing

In [44]:
checkpoint = "distilbert/distilbert-base-cased"

In [45]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [46]:
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )


    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label it (0, 0)
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [47]:
tokenized_squad = ds.map(preprocess_function, batched=True, remove_columns=ds['train'].column_names)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [48]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

In [49]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

model = AutoModelForQuestionAnswering.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForQuestionAnswering LOAD REPORT from: distilbert/distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [52]:
training_args = TrainingArguments(
    output_dir="my_awesome_qa_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad['train'],
    eval_dataset=tokenized_squad['test'],
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,5.762415
2,No log,5.634036
3,No log,5.571373


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=15, training_loss=5.680711364746093, metrics={'train_runtime': 617.1724, 'train_samples_per_second': 0.389, 'train_steps_per_second': 0.024, 'total_flos': 23517558005760.0, 'train_loss': 5.680711364746093, 'epoch': 3.0})

## Evaluate

## Inference